# Begleitendes Notebook zu Kapitel 7
**Build Your First LLM — Kapitel 7: Vorbereitung deiner Daten**
Dieses Notebook führt durch die Schritte der Datenaufbereitung: Bereinigung, Deduplizierung, Aufteilung, Chunking und Speicherung in JSONL mit schnellen Statistiken.
Kleine Beispieldaten sind inline enthalten; es werden keine externen Dateien benötigt.


In [ ]:
# ===== IMPORTS =====
import re           # Reguläre Ausdrücke für Mustererkennung in Text
import json         # JSON- und JSONL-Dateien lesen/schreiben
import hashlib      # Erstellt Fingerabdrücke für die Duplikaterkennung
import unicodedata  # Unicode-Normalisierung (behandelt Sonderzeichen)
from collections import Counter  # Worthäufigkeiten zählen
import random       # Daten für train/val/test-Aufteilung mischen

print('Setup abgeschlossen')

## Python-Tools Kurzreferenz

Dieses Notebook verwendet mehrere Python-Tools. Hier ist eine kurze Anleitung:

**Reguläre Ausdrücke (Regex):** Musterbasiertes Suchen/Ersetzen in Text
- `re.sub(pattern, replacement, text)` — Findet alle Übereinstimmungen von `pattern` und ersetzt sie
- Muster `<[^>]+>` bedeutet: finde `<`, dann beliebige Zeichen ausser `>`, dann `>`  → findet HTML-Tags

**Hashing:** Erstellt einen eindeutigen "Fingerabdruck" für jeden Text
- Gleicher Text → gleicher Hash (immer). Unterschiedlicher Text → unterschiedlicher Hash (fast immer)
- Nützlich zur Duplikaterkennung ohne Vergleich ganzer Dokumente
- `hashlib.sha1(text.encode()).hexdigest()` → 40-Zeichen-Fingerabdruck

**JSON/JSONL:** Datenformate zum Speichern strukturierter Daten
- **JSON:** Eine grosse Datei mit allen Daten (muss die gesamte Datei in den Speicher laden)
- **JSONL:** Ein JSON-Datensatz pro Zeile (kann zeilenweise streamen = speichereffizient)

**Sets:** Sammlungen ohne Duplikate, schnelle "Ist X in dieser Menge?"-Prüfung
- `seen = set()` dann `seen.add(item)` and `item in seen`

**Typhinweise** (`: str`, `: float = 0.8`): Dokumentation für Menschen (Python ignoriert sie)
- `text: str` bedeutet "text sollte ein String sein"
- `train_p: float = 0.8` bedeutet "train_p sollte eine Dezimalzahl sein, Standard ist 0.8"

## Beispieltexte
Ein paar Beispielabsätze, um einen kleinen Korpus zu simulieren.


In [ ]:
raw_docs = [
    "THE TIME MACHINE — CHAPTER I\n\nThis   is   a    sample text… with   odd spacing, smart “quotes”, and tabs\t.",
    "AI systems learn from examples. Data quality shapes model quality.",
    "The key to machine learning is data; the secret to building AI is understanding.",
]
print('docs:', len(raw_docs))


## Bereinigung & Normalisierung

HTML entfernen, Steuerzeichen entfernen, Leerzeichen zusammenfassen, Anführungszeichen normalisieren.


In [ ]:
def clean_text(text: str) -> str:
    """Bereinigt und normalisiert Text für LLM-Training."""
    # Unicode normalization: ﬁ → fi, ｆｕｌｌ → full, etc.
    # NFKC = Kompatibilitätszerlegung + Kanonische Komposition
    text = unicodedata.normalize("NFKC", text)
    
    # Strip HTML tags: <p>, <div>, <span class="foo">, etc.
    # Pattern: < gefolgt von beliebigen Zeichen ausser >, dann >
    text = re.sub(r"<[^>]+>", " ", text)
    
    # Zeilenumbrüche und Tabs durch Leerzeichen ersetzen
    text = re.sub(r"[\n\t]", " ", text)
    
    # Mehrere Leerzeichen zu einem zusammenfassen, führende/nachfolgende Leerzeichen entfernen
    text = re.sub(r"\s+", " ", text).strip()
    
    # Smart Quotes zu geraden Anführungszeichen normalisieren
    text = text.replace("“", '"').replace("”", '"')
    
    return text

cleaned_docs = [clean_text(d) for d in raw_docs]
for i, d in enumerate(cleaned_docs):
    print(f"Cleaned {i}: {d[:80]}...")

## Deduplizierung

Absätze hashen, Wiederholungen entfernen.


In [ ]:
# Mit SHA-1 einen Fingerabdruck für den Text erzeugen
# Gleicher Text → immer derselbe Fingerabdruck
# Anderer Text → mit sehr hoher Wahrscheinlichkeit ein anderer Fingerabdruck
def hash_chunk(text: str) -> str:
    # .encode("utf-8") wandelt String in Bytes um (von hashlib benötigt)
    # .hexdigest() gibt den Hash als 40-Zeichen-String zurück
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

# Hashing demonstrieren
sample = "The cat sat on the mat."
print(f"Text: '{sample}'")
print(f"Hash: {hash_chunk(sample)}")
print(f"Gleicher Text gleicher Hash: {hash_chunk(sample) == hash_chunk(sample)}")
print(f"Unterschiedlicher Text unterschiedlicher Hash: {hash_chunk(sample) != hash_chunk('Dog.')}")

def dedup_chunks(chunks):
    """Entfernt doppelte Chunks mittels Hash-basiertem Fingerprinting."""
    seen = set()    # Verfolgt gesehene Hashes (schnelle Suche!)
    unique = []     # Behält nur eindeutige Chunks
    
    for c in chunks:
        h = hash_chunk(c)
        if h in seen:
            continue        # Duplikat überspringen
        seen.add(h)         # Diesen Hash merken
        unique.append(c)    # Chunk behalten
    
    return unique

# Ein Duplikat hinzufügen und prüfen, ob es entfernt wird
test_docs = cleaned_docs + [cleaned_docs[0]]  # Kopie des ersten Dokuments hinzufügen
deduped = dedup_chunks(test_docs)
print(f'\nVor Deduplizierung: {len(test_docs)} docs')
print(f'Nach Deduplizierung:  {len(deduped)} docs')

## Train/Val/Test-Aufteilung (nach Dokument)

Verwandten Text zusammenhalten, Datenlecks vermeiden.


In [ ]:
def split_docs(docs: list, train_p: float = 0.8, val_p: float = 0.1, seed: int = 42):
    """Teilt docs in Train/Val/Test-Sets auf.
    
    Args:
        docs: Liste der aufzuteilenden docs
        train_p: Anteil für Training (Standard 0.8 = 80%)
        val_p: Anteil für Validierung (Standard 0.1 = 10%)
        seed: Zufalls-Seed für Reproduzierbarkeit
    
    Returns:
        train, val, test Listen (Test bekommt den verbleibenden Anteil)
    """
    # Arbeite mit einer Kopie, um die Liste des Aufrufers nicht zu verändern
    docs = list(docs)
    random.seed(seed)
    random.shuffle(docs)
    
    n = len(docs)
    n_train = int(n * train_p)
    n_val = int(n * val_p)
    
    train = docs[:n_train]
    val = docs[n_train:n_train + n_val]
    test = docs[n_train + n_val:]
    
    # Rundung kann bei winzigen Korpora eine Aufteilung aushungern (3 Dokumente
    # bei 0.34/0.33 ergeben int(3*0.33)=0 für val). Bleibt eine Aufteilung
    # leer, leihe ein Dokument aus der grössten Aufteilung aus, damit jede
    # nach Möglichkeit mindestens eines bekommt.
    if n >= 3:
        for target in (train, val, test):
            if not target:
                donors = sorted((train, val, test), key=len, reverse=True)
                if len(donors[0]) > 1:
                    target.append(donors[0].pop())

    return train, val, test

# Apply split to our deduped docs
train_docs, val_docs, test_docs = split_docs(deduped, train_p=0.6, val_p=0.2, seed=42)
print(f'Train: {len(train_docs)}, Val: {len(val_docs)}, Test: {len(test_docs)}')


## Chunking für Kontextfenster
Langen Text mit Überlappung aufbrechen, um Kontext über Chunk-Grenzen hinweg zu bewahren.


In [ ]:
def chunk_text(text: str, max_chars: int = 200, overlap: int = 50):
    """Zerlegt Text in überlappende Chunks.
    
    Args:
        text: Eingabetext zum Aufteilen
        max_chars: Maximale Zeichen pro Chunk
        overlap: Anzahl der überlappenden Zeichen zwischen Chunks
    
    Returns:
        Liste von Text-Chunks
    """
    assert 0 <= overlap < max_chars, "overlap must be smaller than max_chars"

    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break  # dieser Chunk erreicht das Ende; kein redundanter Rest
        start += max_chars - overlap  # Vorwärts um (max_chars - overlap)
    return chunks

# Wende Chunking auf alle Aufteilungen an
chunked = []
for split, docs in [('train', train_docs), ('val', val_docs), ('test', test_docs)]:
    for d in docs:
        for c in chunk_text(d, max_chars=120, overlap=30):
            chunked.append({'text': c, 'split': split, 'source': 'toy'})
            
print('Gesamt Chunks:', len(chunked))


## Überlappung visualisieren
Sieh, wie Chunks überlappen, um Kontext über Grenzen hinweg zu bewahren.


In [ ]:
# Erstelle einen Testtext mit klaren Positionen
test_text = "A" * 500  # 500 Zeichen
chunks = chunk_text(test_text, max_chars=200, overlap=50)

print(f"Text length: {len(test_text)}")
print(f"Number of chunks: {len(chunks)}")
print(f"Chunk lengths: {[len(c) for c in chunks]}")

# Überlappung zwischen aufeinanderfolgenden Chunks überprüfen
# Python slice notation:
#   text[-50:]  = letzte 50 Zeichen (negativer Index zählt vom Ende)
#   text[:50]   = erste 50 Zeichen
if len(chunks) >= 2:
    # Letzte 50 Zeichen von Chunk 0 sollten den ersten 50 Zeichen von Chunk 1 entsprechen
    chunk0_end = chunks[0][-50:]    # Letzte 50 Zeichen von Chunk 0
    chunk1_start = chunks[1][:50]   # Erste 50 Zeichen von Chunk 1
    overlap_matches = chunk0_end == chunk1_start
    
    print(f"\nOverlap verification: {overlap_matches}")
    print(f"Chunk 0 ends with: ...{chunks[0][-10:]}")
    print(f"Chunk 1 starts with: {chunks[1][:10]}...")
    
# Mit echtem Text
real_text = "This is sentence one. This is sentence two. This is sentence three." * 5
real_chunks = chunk_text(real_text, max_chars=100, overlap=30)
print(f"\nReal text chunked into {len(real_chunks)} pieces")
print(f"Chunk 0: ...{real_chunks[0][-40:]}")
print(f"Chunk 1: {real_chunks[1][:40]}...")
print("\n✅ Überlappung mildert Satzschnitte an Chunk-Grenzen (keine Garantie).")

## Qualitätsprüfungen & Plausibilitätsvalidierung
Probleme früh erkennen mit automatisierten Prüfungen, die leere Chunks, HTML-Lecks und Grössenprobleme markieren.


In [ ]:
def sanity_check(chunks, stage_name):
    """Führt Plausibilitätsprüfungen auf Daten in jeder Pipeline-Phase durch."""
    print(f"\n{'='*50}")
    print(f"Sanity Check: {stage_name}")
    print(f"{'='*50}")
    
    if not chunks:
        print("⚠️  WARNING: No chunks!")
        return
    
    # Grundlegende Statistiken
    print(f"✓ Total chunks: {len(chunks)}")
    lengths = [len(c) if isinstance(c, str) else len(c.get('text', '')) for c in chunks]
    avg_len = sum(lengths) / len(lengths)
    print(f"✓ Avg length: {avg_len:.0f}")
    print(f"✓ Max length: {max(lengths)}")
    print(f"✓ Min length: {min(lengths)}")
    
    # Auf Probleme prüfen
    if max(lengths) > 10 * avg_len:
        print("⚠️  WARNUNG: Maximale Länge ist 10x Durchschnitt - Chunking könnte fehlerhaft sein")
    
    # Auf HTML-Lecks prüfen
    texts = [c if isinstance(c, str) else c.get('text', '') for c in chunks]
    all_text = ' '.join(texts).lower()
    html_words = {'div', 'span', 'href', 'html', 'class', 'src'}
    found_html = [w for w in html_words if w in all_text]
    if found_html:
        print(f"⚠️  WARNUNG: HTML-Tags gefunden: {found_html}")
    else:
        print("✓ Keine HTML-Lecks erkannt")
    
    # Auf leere Chunks prüfen
    empty = sum(1 for l in lengths if l < 10)
    if empty > 0:
        print(f"⚠️  WARNING: {empty} Chunks sind < 10 Zeichen")
    else:
        print("✓ Keine leeren Chunks")
    
    # Beispiel
    sample = texts[0] if texts else "N/A"
    print(f"\n✓ Beispiel: {sample[:100]}...")
    print()

# Führe Prüfungen auf unseren aufgeteilten Daten aus
sanity_check(chunked, "Nach Chunking")

# Du kannst dies nach jeder Phase ausführen:
# sanity_check(cleaned_docs, "Nach Bereinigung")
# sanity_check(deduped, "Nach Deduplizierung")


## Durchgearbeitetes Beispiel: End-to-End-Pipeline
Vollständiger Durchgang von Rohdokumenten (mit HTML und Duplikaten) zu JSONL-fähigen Daten. Das Notebook speichert drei englische Beispielsätze in `train.jsonl`; das Buch verwendet drei deutsche Sätze. Beide Dateien haben dieselben Felder.


In [ ]:
print("="*60)
print("VOLLSTÄNDIGE DATENPIPELINE-DURCHFÜHRUNG")
print("="*60)

# Schritt 1: Beginne mit Rohdokumenten (unordentlich, mit Duplikaten und HTML)
print("\n📥 SCHRITT 1: Rohdokumente")
raw_pipeline_docs = [
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc1"},
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc2"},  # exaktes Duplikat!
    {"text": "<p>The dog    ran\tin the park.</p>", "source": "doc3"},
    {"text": "The bird flew over the house.", "source": "doc4"}
]
print(f"   Rohdokumente: {len(raw_pipeline_docs)}")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text'][:50]}...")

# Schritt 2: Jedes Dokument bereinigen
print("\n🧹 SCHRITT 2: Bereinigung")
for doc in raw_pipeline_docs:
    doc["text"] = clean_text(doc["text"])
print("   HTML entfernt, Leerzeichen normalisiert")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text']}")

# Beim Deduplizieren bleibt die Quelle des ersten Dokuments erhalten.
unique_pipeline, seen = [], set()
for doc in raw_pipeline_docs:
    fingerprint = hash_chunk(doc['text'])
    if fingerprint not in seen:
        seen.add(fingerprint)
        unique_pipeline.append(doc)

train_p, val_p, test_p = split_docs(unique_pipeline, train_p=0.34, val_p=0.33, seed=42)
final_records = []
for split, docs in [('train', train_p), ('val', val_p), ('test', test_p)]:
    print(split, [doc['source'] for doc in docs])
    for doc in docs:
        for chunk in chunk_text(doc['text'], max_chars=50, overlap=10):
            final_records.append({'text': chunk, 'source': doc['source'],
                                  'split': split, 'length': len(chunk)})
print('Beispieldatensatz:', final_records[0])
print('Drei kurze Datensätze zeigen das Format, nicht die Menge für ein Modelltraining.')


In [ ]:
def save_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

save_jsonl(final_records, 'train.jsonl')
print('train.jsonl geschrieben mit', len(final_records), 'records')


## Schnelle Statistiken

Duplikatrate, häufigste Wörter und Beispieldatensätze.


In [ ]:
def duplicate_rate(texts):
    hashes = [hash_chunk(t) for t in texts]
    return 1 - (len(set(hashes)) / len(hashes))

def top_words(texts, k=10):
    words = " ".join(texts).lower().split()
    return Counter(words).most_common(k)

texts_all = [r['text'] for r in final_records]
print('Duplikatrate:', duplicate_rate(texts_all))
print('Häufigste Wörter:', top_words(texts_all, k=8))
print('Beispiel records:', final_records[:2])


## Zusammenfassung

In diesem Notebook hast du die vollständige Datenpipeline für LLM-Training gelernt:

**1. Textbereinigung:** HTML entfernen, Leerzeichen normalisieren, Sonderzeichen behandeln
**2. Deduplizierung:** Hashing verwenden, um exakte Duplikate zu identifizieren und zu entfernen
**3. Train/Val/Test-Aufteilung:** Daten auf Dokumentebene trennen, um Datenlecks zu verhindern
**4. Chunking mit Überlappung:** Lange Texte in LLM-geeignete Stücke aufbrechen und dabei Kontext bewahren
**5. Qualitätsprüfungen:** Automatisierte Plausibilitätsprüfungen erkennen Probleme frühzeitig
**6. JSONL-Format:** Daten in einem streaming-freundlichen Format speichern

**Schlüsselkonzepte:**
- **Überlappung** bewahrt Kontext über Chunk-Grenzen hinweg (mildert das Zerhacken von Sätzen; keine Garantie)
- **Hashing** liefert schnelle, deterministische Fingerabdrücke für Deduplizierung
- **Aufteilung auf Dokumentebene** hält verwandte Chunks zusammen in derselben Aufteilung
- **Qualitätsprüfungen** erkennen HTML-Lecks, leere Chunks und Grössenanomalien, bevor sie Trainingsprobleme verursachen

**Nächste Schritte:**
- Kapitel 8: Tokenisierung (Text in Zahlen umwandeln)
- Auf echte Datensätze skalieren (Wikipedia, Common Crawl, Bücher)
- Mit verschiedenen Überlappungswerten für deinen Anwendungsfall experimentieren

Die Datenpipeline ist die Grundlage jedes grossartigen LLM!
